In [ ]:
# 导入APIKEY
from dotenv import find_dotenv, load_dotenv

_ = load_dotenv(find_dotenv())

In [ ]:
# 抓取文件夹路径

from pathlib import Path
import subprocess
import os

folder_path = "llm-universe/data_base/knowledge_db"
files_path = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        files_path.append(file_path)

In [3]:
# 加载每个文件（Langchain）
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader

loaders = []
for file_path in files_path:
    tag = file_path.split(".")[-1].lower()
    if (tag == 'pdf'):
        loaders.append(PyMuPDFLoader(file_path))
    if (tag == 'md'):
        loaders.append(UnstructuredMarkdownLoader(file_path))

texts = []
for loader in loaders:
    texts.extend(loader.load())


In [4]:
# 切成chunks（Langchain）
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

split_text = text_splitter.split_documents(texts)

In [ ]:
# 向量化（Langchain）
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
import shutil

embedding = OpenAIEmbeddings(
    model='text-embedding-3-small'
)

persist_dir = "llm-universe/data_base/vector_db/chroma"

if not os.path.exists(persist_dir):
    vectordb = Chroma(
        embedding_function=embedding,
        persist_directory=persist_dir
    )

    batch_size = 50
    for i in range(0,  len(split_text), batch_size):
        docs = split_text[i: i + batch_size]
        vectordb.add_documents(docs)
            
    vectordb.persist()

# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings

# embedding = OpenAIEmbeddings(
#     model="text-embedding-3-small"
# )

# PROJECT_ROOT = Path(
#     subprocess.check_output(
#         ["git", "rev-parse", "--show-toplevel"],
#         text=True
#     ).strip()
# )

# persist_dir = PROJECT_ROOT / "Naive RAG" / "data_base" / "vector_db" / "chroma"

# vectordb = Chroma(
#     persist_directory=str(persist_dir),
#     embedding_function=embedding
# )



2026-05-29 15:57:16.228930638 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [6]:
# 制作retriever（Langchain）
from langchain_core.runnables import RunnableLambda
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

def combine(docs):
    return "\n\n".join(doc.page_content for doc in docs)

combiner = RunnableLambda(combine)

vector_retriever = vectordb.as_retriever(
    search_kwargs={"k": 10}
    )

bm_retriever = BM25Retriever.from_documents(
    split_text,
)
bm_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm_retriever],
    weights=[0.5, 0.5]
)

reranker = CohereRerank(
    model="rerank-v3.5",
    top_n=8
)


retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_retriever,
    base_compressor=reranker
)

retriever_chain = retriever | combiner


In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.1",
    temperature=0
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch
from langchain_core.output_parsers import StrOutputParser


condense_question_system_template = (

    "请根据聊天记录和用户最新问题，"

    "把用户最新问题改写成一个可以独立理解的问题。"

    "不要回答问题，只需要返回改写后的问题。"

)

condense_question_prompt = ChatPromptTemplate.from_messages([
    ("system", condense_question_system_template),

    ("placeholder", "{chat_history}"),

    ("human", "{input}"),
])

retriever_docs = RunnableBranch(
    (lambda x : not x.get("chat_history", False), RunnableLambda(lambda x : x["input"]) | retriever),
    condense_question_prompt | llm | StrOutputParser() | retriever
)

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

system_prompt = (
    "你是一个问答任务的助手。"
    "请使用检索到的上下文片段回答问题。"
    "如果你不知道答案就说不知道。"
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

qa_chain = (
    RunnablePassthrough.assign(
        context=lambda x: combine(x["context"])
    )
    |qa_prompt
    |llm
    |StrOutputParser()
)

qa_history_chain = (
    RunnablePassthrough.assign(
        context=retriever_docs
    ).assign(
        answer=qa_chain
    )
)

In [10]:
res = qa_history_chain.invoke({
    "input": "它的作者是谁？",
    "chat_history": [
        ("human", "西瓜书是什么？"),
        ("ai", "西瓜书通常指周志华老师的《机器学习》一书。")
    ]
})

print(res["answer"])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


《机器学习》（“西瓜书”）的作者是 周志华。


In [11]:
from app.rag_chain import create_rag_chain

qa_history_chain = create_rag_chain()

res = qa_history_chain.invoke({
    "input": "它的作者是谁？",
    "chat_history": [
        ("human", "西瓜书是什么？"),
        ("ai", "西瓜书通常指周志华老师的《机器学习》一书。")
    ]
})

print(res["answer"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


《西瓜书》（《机器学习》）的作者是 **周志华**。
